In [2]:
import pandas as pd

# 1. Charger le excel
df = pd.read_excel("C:/Users/Moi/Documents/R3_WEB.xlsx")
df

In [4]:
# 5. Création de la colonne PhraseContextuelle sans séparateurs en trop
df['PhraseContextuelle'] = df[["Libellé WEB", "Univers", "Rayon"]] \
    .fillna('') \
    .agg(lambda x: "|".join([v.strip() for v in x if v.strip() != ""]), axis=1)

# 6. Export du fichier propre
df.to_excel("C:/Users/Moi/Documents/catalogue_web.xlsx", index=False)

In [9]:
!pip uninstall torch torchvision torchaudio -y
!pip cache purge

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0


Files removed: 2210


In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --no-cache-dir --timeout 1000

Note: you may need to restart the kernel to use updated packages.Looking in indexes: https://download.pytorch.org/whl/cu118
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 3.6 MB/s eta 0:12:56
   ---------------------------------------- 0.0/2.8 GB 3.2 MB/s eta 0:14:55
   ---------------------------------------- 0.0/2.8 GB 3.0 MB/s eta 0:15:30
   ---------------------------------------- 0.0/2.8 GB 3.0 MB/s eta 0:15:28
   ---------------------------------------- 0.0/2.8 GB 3.1 MB/s eta 0:15:13
   ---------------------------------------- 0.0/2.8 GB 3.2 MB/s eta 0:14:42
   ---------------------------------------- 0.0/2.8 GB 3.1 MB/s eta 0:15:07
   ---------------------------------------- 0.0/2.8 GB 3.1 MB/s eta 0:15:01
   ---------------------------------------- 0.0/2.8 GB 3.1 MB/s eta 0:15:01
   ---------------------------------------- 0.0/2.8 GB 3.1 MB/s eta 0:14:57
   ---------------------------------------- 0.0

   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:10
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:14
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:14
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:14
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:14
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:10
   ---- ----------------------------------- 0.3/2.8 GB 3.0 MB/s eta 0:14:06
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:14
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:14
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:21
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:22
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:22
   ---- ----------------------------------- 0.3/2.8 GB 2.9 MB/s eta 0:14:22
   ---- ----

In [1]:
print(torch.cuda.is_available())

NameError: name 'torch' is not defined

In [ ]:
import pandas as pd
import numpy as np
import torch
import faiss
import time
from sentence_transformers import SentenceTransformer
import os

# ===========================
# CONFIGURATION
# ===========================
MODEL_NAME = "all-mpnet-base-v2"
INPUT_FILE = "catalogue_web.xlsx"  # ⚠️ ton nouveau fichier
OUTPUT_PREFIX = "ugap_mnet"

CHUNK_SIZE = 50000
BATCH_SIZE = 128  # GPU si possible

# ===========================
# 1️⃣ CHARGEMENT DES DONNÉES
# ===========================
print(f"📥 Lecture du fichier : {INPUT_FILE}")
df_web = pd.read_excel(INPUT_FILE, dtype={"Reference": str})

if "PhraseContextuelle" not in df_web.columns:
    raise ValueError("❌ 'PhraseContextuelle' manquante")

# Nettoyage
df_web["PhraseContextuelle"] = (
    df_web["PhraseContextuelle"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

phrases = df_web["PhraseContextuelle"].tolist()
print(f"🧱 Produits à encoder : {len(phrases):,}")

# ===========================
# 2️⃣ CHARGEMENT MODÈLE
# ===========================
print("\n🚀 Chargement du modèle...")
start_time = time.time()

model = SentenceTransformer(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"✅ Device : {device.upper()}")

# ===========================
# 3️⃣ ENCODAGE DIRECT (SANS CHUNKS DISQUE)
# ===========================
print("\n⚙️ Encodage en cours...")

embeddings_list = []

for i in range(0, len(phrases), CHUNK_SIZE):
    chunk = phrases[i:i+CHUNK_SIZE]

    print(f"➡️ Chunk {i//CHUNK_SIZE + 1} ({len(chunk)} phrases)")

    emb = model.encode(
        chunk,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=BATCH_SIZE,
        show_progress_bar=True
    ).astype("float32")

    embeddings_list.append(emb)

# Fusion
embeddings = np.vstack(embeddings_list)

print(f"✅ Encodage terminé : {embeddings.shape}")

# ===========================
# 4️⃣ SAUVEGARDE
# ===========================
np.save(f"embeddings_{OUTPUT_PREFIX}.npy", embeddings)
df_web.to_parquet(f"produits_{OUTPUT_PREFIX}.parquet", index=False)

print("💾 Fichiers sauvegardés")

# ===========================
# 5️⃣ INDEX FAISS
# ===========================
print("\n📦 Création index FAISS...")

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

faiss.write_index(index, f"index_{OUTPUT_PREFIX}.faiss")

print("✅ Index FAISS créé")

# ===========================
# 6️⃣ TEMPS TOTAL
# ===========================
elapsed = time.time() - start_time
mins, secs = divmod(elapsed, 60)

print("\n🎯 TERMINÉ")
print(f"Temps : {int(mins)} min {int(secs)} sec")